# Workshop Setup and Verification

**Time**: 15 minutes

This notebook will help you:
- Install required packages
- Configure credentials
- Verify Elastic Cloud connection
- Verify AWS Bedrock access
- Test ELSER model

---

## Step 1: Install Required Packages

Run this cell to install all dependencies:

In [ ]:
# Install packages
!pip install -q elasticsearch==8.15.0 elastic-apm==6.21.0 boto3==1.34.70 python-dotenv==1.0.1 requests==2.31.0

print("✅ All packages installed successfully!")

## Step 2: Configure Credentials

**Edit the values below with your actual credentials:**

In [ ]:
import os

# Elastic Cloud Configuration
os.environ['ELASTIC_CLOUD_ID'] = 'your-cloud-id-here'
os.environ['ELASTIC_USERNAME'] = 'elastic'
os.environ['ELASTIC_PASSWORD'] = 'your-password-here'
os.environ['ELASTIC_ENDPOINT'] = 'https://your-deployment.es.us-east-1.aws.found.io:9243'

# AWS Configuration
os.environ['AWS_REGION'] = 'us-east-1'
os.environ['AWS_ACCESS_KEY_ID'] = 'your-access-key'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key'

# Strands API (optional for initial testing)
os.environ['STRANDS_API_KEY'] = 'your-strands-key-or-leave-blank'

print("✅ Credentials configured!")
print("\n⚠️  Remember: Never commit this notebook with real credentials!")

## Step 3: Test Elasticsearch Connection

Let's verify we can connect to your Elastic Cloud deployment:

In [ ]:
from elasticsearch import Elasticsearch

# Connect to Elastic Cloud
es = Elasticsearch(
    cloud_id=os.getenv('ELASTIC_CLOUD_ID'),
    basic_auth=(os.getenv('ELASTIC_USERNAME'), os.getenv('ELASTIC_PASSWORD'))
)

# Get cluster info
info = es.info()

print("✅ Successfully connected to Elastic Cloud!")
print(f"\nCluster Name: {info['cluster_name']}")
print(f"Version: {info['version']['number']}")
print(f"Cluster UUID: {info['cluster_uuid']}")

## Step 4: Verify ELSER Model

Check if ELSER v2 is deployed and running:

In [ ]:
try:
    # Check ELSER model
    response = es.ml.get_trained_models(model_id=".elser_model_2")
    
    if response['trained_model_configs']:
        print("✅ ELSER model found!")
        
        # Check deployment status
        stats = es.ml.get_trained_models_stats(model_id=".elser_model_2")
        
        if stats['trained_model_stats']:
            deployment = stats['trained_model_stats'][0].get('deployment_stats', {})
            state = deployment.get('state', 'unknown')
            
            if state == 'started':
                print("✅ ELSER is deployed and running!")
                print(f"\nAllocations: {deployment.get('number_of_allocations', 0)}")
                print(f"Threads per allocation: {deployment.get('threads_per_allocation', 0)}")
            else:
                print(f"⚠️  ELSER state: {state}")
                print("Please start the model in Kibana → ML → Trained Models")
        else:
            print("⚠️  ELSER model exists but not deployed")
            print("Please deploy in Kibana → ML → Trained Models")
    else:
        print("❌ ELSER model not found")
        print("Please download and deploy ELSER in Kibana → ML → Trained Models")
        
except Exception as e:
    print(f"❌ Error checking ELSER: {e}")
    print("\nMake sure:")
    print("1. Your Elastic deployment has ML enabled")
    print("2. ELSER model is downloaded")
    print("3. ELSER model is deployed and started")

## Step 5: Test AWS Bedrock Access

Verify you have access to Claude models:

In [ ]:
import boto3

try:
    # Create Bedrock client
    bedrock = boto3.client('bedrock', region_name=os.getenv('AWS_REGION'))
    
    # List available models
    models = bedrock.list_foundation_models()
    
    # Find Claude models
    claude_models = [
        m for m in models['modelSummaries'] 
        if 'claude' in m['modelId'].lower()
    ]
    
    print(f"✅ AWS Bedrock access verified!")
    print(f"\nAvailable Claude models: {len(claude_models)}")
    
    for model in claude_models[:3]:  # Show first 3
        print(f"  • {model['modelId']}")
        
    if len(claude_models) == 0:
        print("\n⚠️  No Claude models accessible")
        print("Please request model access in AWS Bedrock console")
        
except Exception as e:
    print(f"❌ Error accessing Bedrock: {e}")
    print("\nMake sure:")
    print("1. AWS credentials are correct")
    print("2. You have Bedrock permissions")
    print("3. Model access has been granted")

## Step 6: Test Simple Query

Let's do a quick test of Claude via Bedrock:

In [ ]:
import json

try:
    bedrock_runtime = boto3.client(
        'bedrock-runtime',
        region_name=os.getenv('AWS_REGION')
    )
    
    # Simple test prompt
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 100,
        "messages": [
            {
                "role": "user",
                "content": "Say 'Hello from Claude!' and confirm you're ready to help with travel planning."
            }
        ]
    }
    
    response = bedrock_runtime.invoke_model(
        modelId="anthropic.claude-3-5-sonnet-20240620-v1:0",
        body=json.dumps(request_body)
    )
    
    response_body = json.loads(response['body'].read())
    message = response_body['content'][0]['text']
    
    print("✅ Claude is responding!\n")
    print(f"Claude says: {message}")
    
except Exception as e:
    print(f"❌ Error calling Claude: {e}")

## ✅ Verification Complete!

If all cells above show ✅, you're ready to proceed!

### Next Steps:

1. **Continue to Notebook 01** - ELSER Semantic Search
2. **Or** review any ❌ errors above and fix them first

---

### Troubleshooting Guide

**Elastic Connection Issues:**
- Verify Cloud ID is correct (no extra spaces)
- Check password is correct
- Ensure deployment is running in Elastic Cloud console

**ELSER Issues:**
- Go to Kibana → ML → Trained Models
- Find `.elser_model_2`
- Click "Download" if not downloaded
- Click "Deploy" if not deployed
- Click "Start" if not started

**AWS Bedrock Issues:**
- Go to AWS Bedrock console → Model access
- Request access to Claude models
- Wait 2-5 minutes for approval
- Verify IAM permissions include `bedrock:*`

---

**Ready?** Open `01-ELSER-Semantic-Search.ipynb` →